# Fooocus in Google Colab

Wähle unter **Laufzeit → Laufzeittyp ändern** eine **GPU** und die aktuelle Laufzeit mit Python 3.13.

**Im Alltag:** Führe **Konfiguration** und anschließend **Start** aus. Das gilt auch für eine neue Colab-Laufzeit: Start stellt die gespeicherten Abhängigkeiten aus Drive wieder her und installiert keine Pakete.

**Beim ersten Mal oder wenn Start einen unpassenden Cache meldet:** Führe nach der Konfiguration einmal **Einrichtung/Cache erstellen (einmalig)** aus, danach **Start**. Bei fehlendem oder unpassendem Cache erklärt Start, wie du die Einrichtung erneuerst. Die Einrichtung ist im normalen Ablauf nicht nötig; führe die Zellen gezielt aus.

Die Anwendung und verwendete Modelle laufen lokal unter `/content/Fooocus`, die Abhängigkeiten unter `/content/fooocus-deps`. Drive speichert Modelle unter `MyDrive/Fooocus/models`, fertige Bilder unter `MyDrive/Fooocus/outputs` und den Paket-Cache unter `MyDrive/Fooocus/cache/python`. Zusätzliche Checkpoints gehören in `models/checkpoints`, LoRAs in `models/loras`; sie werden bei Verwendung auf die lokale Festplatte kopiert. Vorhandene Drive-Dateien bleiben erhalten.

Die erste Verwendung eines Modells kann einen Download erfordern. Lösche lokale Ausgabedateien erst, wenn sie erfolgreich auf Drive gespeichert wurden. Ohne Drive bleibt der Cache nur für die aktuelle Laufzeit verfügbar.

Eine vollständige Bildgenerierung auf einer echten Colab-GPU gehört weiterhin zu den dokumentierten Abnahmeprüfungen.


## Konfiguration
Drive verbinden und den Quelltext aktualisieren. Diese Zelle installiert keine Pakete.


In [ ]:
import subprocess, sys
from pathlib import Path

REPOSITORY = "https://github.com/abjilox/Fooocus.git"
BRANCH = "better-colab"
LOCAL_REPO = Path("/content/Fooocus")
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/Fooocus" #@param {type:"string"}
STATE_FILE = "/content/fooocus-colab-state.json"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

def git_output(*args, show_output=False):
    result = subprocess.run(["git", *args], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    if show_output or result.returncode:
        print(result.stdout, end="", flush=True)
    result.check_returncode()
    return result.stdout.strip()

if not LOCAL_REPO.exists():
    git_output("clone", "--branch", BRANCH, "--single-branch", REPOSITORY,
               str(LOCAL_REPO), show_output=True)
else:
    if not (LOCAL_REPO / ".git").exists():
        raise RuntimeError(f"{LOCAL_REPO} ist kein Git-Arbeitsverzeichnis. Wähle einen anderen lokalen Ordner.")
    git_args = ("-C", str(LOCAL_REPO))
    if git_output(*git_args, "remote", "get-url", "origin").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("Der lokale Ordner gehört zu einem anderen Repository. Wähle einen anderen LOCAL_REPO.")
    if git_output(*git_args, "branch", "--show-current") != BRANCH:
        raise RuntimeError(f"Der lokale Ordner verwendet nicht den Branch {BRANCH}. Wähle einen anderen lokalen Ordner.")
    if git_output(*git_args, "status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError("Lokale Quelltextänderungen gefunden. Sichere sie vor dem Aktualisieren; es wurde nichts überschrieben.")
    git_output(*git_args, "fetch", "origin", BRANCH, show_output=True)
    git_output(*git_args, "merge", "--ff-only", "FETCH_HEAD", show_output=True)

sys.path.insert(0, str(LOCAL_REPO))
from modules.colab_process import run_live

BOOTSTRAP_COMMAND = [sys.executable, "-u", str(LOCAL_REPO / "scripts/colab_bootstrap.py")]
DRIVE_ARGUMENTS = ["--drive-root", DRIVE_ROOT] if USE_GOOGLE_DRIVE else []
print("Konfiguration bereit. Normalerweise jetzt die Zelle Start ausführen.", flush=True)


## Einrichtung/Cache erstellen (einmalig)
Nur für die erste Einrichtung oder zum Erneuern eines unpassenden Cache. Fehlende Abhängigkeiten werden installiert und für spätere Laufzeiten gespeichert. Ein passender Cache wird direkt wiederverwendet. Aktiviere `REBUILD_CACHE` nur, wenn du den Cache ausdrücklich neu erstellen möchtest.


In [ ]:
# Nur beim ersten Mal oder wenn Start einen fehlenden/unpassenden Cache meldet.
REBUILD_CACHE = False #@param {type:"boolean"}
run_live(BOOTSTRAP_COMMAND + ["prepare", "--state-file", STATE_FILE] + DRIVE_ARGUMENTS
         + (["--rebuild"] if REBUILD_CACHE else []), cwd=str(LOCAL_REPO))


## Start
Nach der Konfiguration direkt ausführen. Ein passender Drive-Cache wird auch ohne lokale Zustandsdatei wiederhergestellt. Hier werden keine Pakete installiert. Öffne die unten ausgegebene Gradio-Adresse. Zum erneuten Starten in derselben Laufzeit genügt diese Zelle.


In [ ]:
PRESET = "default" #@param ["default", "anime", "realistic"]
run_live(BOOTSTRAP_COMMAND + ["start", "--state-file", STATE_FILE, "--preset", PRESET] + DRIVE_ARGUMENTS,
         cwd=str(LOCAL_REPO))
